In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "sst2"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence', 'label', 'idx'],
    num_rows: 67349
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence': 'hide new secretions from the parental units ',
 'label': 0,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence']

"that 's far too tragic to merit such superficial treatment "

In [11]:
raw_train_dataset[5]['label']

0

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence': Value('string'),
 'label': ClassLabel(names=['negative', 'positive']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1821
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 47]),
 'token_type_ids': torch.Size([32, 47]),
 'attention_mask': torch.Size([32, 47])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(0.7323, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda', index=1)

### Load Trained Weights

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt')
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
from train_eval_func import eval_loop

In [28]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.9311926605504587}

# 5. Merge Most Similar Layers

In [29]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from tqdm import tqdm

In [30]:
import os
from merge_helper import merge_bert_layers, LayerMergeTracker
from train_eval_func import train, get_primary_metric, set_lr_scheduler, iterative_cka_merge_and_train, get_recovery_epoch
from torch.optim import AdamW

In [31]:
init_metric = eval_loop(model, val_dataloader, task_name, device)[0]
init_metric

{'accuracy': 0.9311926605504587}

In [32]:
tracker = LayerMergeTracker(num_layers=12)

print("Initial state:")
print(f"  {tracker.get_mapping()}")
print(f"  Total layers: {len(tracker)}")
print()

Initial state:
  [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11]]
  Total layers: 12



In [33]:
from CKA import CKAEvaluator

In [34]:
cka_eval = CKAEvaluator(device)

In [35]:
recovery_epochs = get_recovery_epoch(task_name)
recovery_epochs

1

In [36]:
history = iterative_cka_merge_and_train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    task_name=task_name,
    device=device,
    cka_evaluator=cka_eval,
    tracker=tracker,
    init_metric=init_metric,
    num_merges=6,
    target_layers=[6, 7, 8],  # Save models with 6, 7, 8 layers
    recovery_epochs=recovery_epochs,
    recovery_lr=1e-5,
    save_dir='./weights/',
    cka_max_iter=150
)

Target Metric for sst2: accuracy
Original accuracy score: 0.9312
Recovery threshold: 0.0093

Merge Iteration: 1/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|██████████| 150/150 [00:07<00:00, 19.07it/s]


  Similarity Stats:
    Average: 0.066280
    Highest: 0.070618
    Lowest:  0.060345

[2/5] Merging Layers...
  Merged Layers: 10 + 11
  Layer Composition: [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10, 11]]
  Total Layers: 11

[3/5] Evaluating Post-Merge Performance...
  Metrics:
    ★ accuracy: 0.9266

[4/5] Recovery Training...
  Performance drop: 0.0046 (threshold: 0.0093)
  → Recovery training NOT needed

[5/5] Saving Model...
  • 11 layers (not in target list, skipping save)

Merge Iteration: 2/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|██████████| 150/150 [00:06<00:00, 21.96it/s]


  Similarity Stats:
    Average: 0.065651
    Highest: 0.068996
    Lowest:  0.059678

[2/5] Merging Layers...
  Merged Layers: 9 + 10
  Layer Composition: [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9, 10, 11]]
  Total Layers: 10

[3/5] Evaluating Post-Merge Performance...
  Metrics:
    ★ accuracy: 0.9220

[4/5] Recovery Training...
  Performance drop: 0.0092 (threshold: 0.0093)
  → Recovery training NOT needed

[5/5] Saving Model...
  • 10 layers (not in target list, skipping save)

Merge Iteration: 3/6

[1/5] Computing CKA Similarity...


...

✓ Saved checkpoint (best loss: 0.4211)

<----------------- Epoch 1 ----------------->
Loss: 0.1, Training Metrics:
  accuracy: 0.9751
Validation Loss: 0.42, Validation Metrics:
  accuracy: 0.9174
Elapsed Time: 130.6814 sec

Total Training Time: 142.72 sec
Chosen Primary Metric: accuracy
Best Metric Run: 0.9174

  → Loading best checkpoint...
  ✓ Loaded best checkpoint from epoch 1
    Best val loss: 0.4211
    Best val metrics: {'accuracy': 0.9174311926605505}
  ✓ Cleaned up temporary checkpoint

[5/5] Saving Model...
  ✓ Saved 7-layer model to ./weights/layer-7-sst2.pt

Merge Iteration: 6/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|██████████| 150/150 [00:04<00:00, 33.51it/s]


  Similarity Stats:
    Average: 0.061432
    Highest: 0.067238
    Lowest:  0.053501

[2/5] Merging Layers...
  Merged Layers: 1 + 2
  Layer Composition: [[0], [1, 2, 3], [4, 5], [6, 7], [8], [9, 10, 11]]
  Total Layers: 6

[3/5] Evaluating Post-Merge Performance...
  Metrics:
    ★ accuracy: 0.9151

[4/5] Recovery Training...
  Performance drop: 0.0161 (threshold: 0.0093)
  → Recovery training NEEDED
  → Training: 1 epochs, lr=1e-05


Training: 100%|██████████| 2105/2105 [02:21<00:00, 14.87it/s]


✓ Saved checkpoint (best loss: 0.4193)

<----------------- Epoch 1 ----------------->
Loss: 0.11, Training Metrics:
  accuracy: 0.9720
Validation Loss: 0.42, Validation Metrics:
  accuracy: 0.9140
Elapsed Time: 129.4888 sec

Total Training Time: 141.52 sec
Chosen Primary Metric: accuracy
Best Metric Run: 0.9140

  → Loading best checkpoint...
  ✓ Loaded best checkpoint from epoch 1
    Best val loss: 0.4193
    Best val metrics: {'accuracy': 0.9139908256880734}
  ✓ Cleaned up temporary checkpoint

[5/5] Saving Model...
  ✓ Saved 6-layer model to ./weights/layer-6-sst2.pt

MERGE LOOP COMPLETED
Total merges: 6
Final layers: 6
Final composition: [[0], [1, 2, 3], [4, 5], [6, 7], [8], [9, 10, 11]]


In [37]:
# See what happened
print("Merge history:")
for i, iteration in enumerate(history['iterations']):
    print(f"  Iteration {iteration['iteration']}:")
    print(f"    Merged: {iteration['merged_layers']}")
    print(f"    Layers after: {iteration['num_layers_after']}")
    print(f"    Composition: {history['layer_compositions'][i]}")
    
    if history['training_stats'][i]:
        final_metrics = history['metrics_after_recovery'][i]
        print(f"    Final metrics: {final_metrics}")

Merge history:
  Iteration 1:
    Merged: (10, 11)
    Layers after: 11
    Composition: [[0], [1, 2, 3], [4, 5], [6, 7], [8], [9, 10, 11]]
  Iteration 2:
    Merged: (9, 10)
    Layers after: 10
    Composition: [[0], [1, 2, 3], [4, 5], [6, 7], [8], [9, 10, 11]]
  Iteration 3:
    Merged: (1, 2)
    Layers after: 9
    Composition: [[0], [1, 2, 3], [4, 5], [6, 7], [8], [9, 10, 11]]
  Iteration 4:
    Merged: (3, 4)
    Layers after: 8
    Composition: [[0], [1, 2, 3], [4, 5], [6, 7], [8], [9, 10, 11]]
  Iteration 5:
    Merged: (4, 5)
    Layers after: 7
    Composition: [[0], [1, 2, 3], [4, 5], [6, 7], [8], [9, 10, 11]]
    Final metrics: {'accuracy': 0.9174311926605505}
  Iteration 6:
    Merged: (1, 2)
    Layers after: 6
    Composition: [[0], [1, 2, 3], [4, 5], [6, 7], [8], [9, 10, 11]]
    Final metrics: {'accuracy': 0.9139908256880734}
